In [ ]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt

import matplotlib.patches as mpatches
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")


In [ ]:
raw_df=pd.read_csv("messy_sales_data.csv")


In [ ]:
raw_df.head()

,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [ ]:
# check1  Missing value

print(f"Missing value:{raw_df.isna().sum()}")

#check2  Duplicate value

print(f"Duplicate value:{raw_df.duplicated().sum()}")

#check3  data types
print(f"DataTypes:{raw_df.dtypes}")

#check4 Unique Values
print(f"Uniquevalues:{raw_df['category'].dropna().unique().tolist()}")
print(f"Sample dataset:{raw_df['order_date'].unique()[:8].tolist()}")
print(f"(Sample names:{raw_df['customer_name'].dropna().unique()[:8].tolist()})")

Missing value:order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
Duplicate value:0
DataTypes:order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object
Uniquevalues:['Electronics', 'Accessories']
Sample dataset:['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15']
(Sample names:['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'Ananya Das', 'Vikram Iyer'])


Create a working copy for ETL

In [ ]:
df=raw_df.copy()

print(f"Working copy created:{df.shape}")


Working copy created:(30, 9)


In [ ]:
print(f"Total Msiing Values:{raw_df.isna().sum().sum()}")

Total Msiing Values:7


In [ ]:
# Fix missing cutomer name
df['customer_name'].fillna('UnknownCustomer',inplace=True)  #inpcae=True applies the chnage directly to df(no need to assign)

print(f"{df.isna().sum().sum()}")
print(df['customer_name'].isna().sum())


5
0


In [ ]:
df['quantity'].fillna(df['quantity'].median(),inplace=True)
print(f"{df.isna().sum().sum()}")
print(df['quantity'].isna().sum())

2
0


In [ ]:
df['category'].fillna('Uncategorized',inplace=True)
df['product'].fillna('NULL',inplace=True)

In [ ]:
print(f"{df.isna().sum().sum()}")

0


In [ ]:
print(f"Total Duplicated values after fix: {df.duplicated().sum()}")

Total Duplicated values after fix: 0


In [ ]:
#Before Duplication
print(f"Rows before duplication:{len(df)}")
print(f"Duplicates rows found :{df.duplicated().sum()}")



#Removal of duplication
#keep = False to marks ALL copies of duplicate as True
duplicate_mask = df.duplicated(keep=False)



print("\nThe duplicate rows(all copies shown):")
print(df[duplicate_mask][['order_id','customer_name','product','order_date']].to_string(index=False))


df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)
print(f"\nRows after duplication:{len(df)}")
print(f"Rows removed:{len(df[duplicate_mask])}")

Rows before duplication:30
Duplicates rows found :0

The duplicate rows(all copies shown):
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

Rows after duplication:30
Rows removed:0


In [ ]:
print("sample data before parsing")
print(df['order_date'].head(10).tolist())

#pd.to_datetime() converts text dates into prope dateime objects
#wonce conevrted we can extract the year ,month,day,day-of-week etc

#errors='coerce' isvery important
# If a date sting cannot be parsed ,instead of crashing it insert NaT
#-Nat menas  "Not a time" - the datetime equivalent a Nan
#-this prevent one bad date from breaking the entire application pipeline

df['order_date']=pd.to_datetime(df['order_date'],dayfirst=False,errors='coerce')

print(df['order_date'].tolist())

sample data before parsing
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13', '2024-01-15', '2024-01-15']
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-08 00:00:00'), Timestamp('2024-01-10 00:00:00'), Timestamp('2024-01-05 00:00:00'), NaT, Timestamp('2024-01-12 00:00:00'), Timestamp('2024-01-13 00:00:00'), Timestamp('2024-01-15 00:00:00'), Timestamp('2024-01-15 00:00:00'), Timestamp('2024-01-18 00:00:00'), Timestamp('2024-01-20 00:00:00'), Timestamp('2024-01-22 00:00:00'), NaT, Timestamp('2024-01-25 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-28 00:00:00'), Timestamp('2024-01-30 00:00:00'), Timestamp('2024-02-01 00:00:00'), Timestamp('2024-02-03 00:00:00'), Timestamp('2024-02-05 00:00:00'), Timestamp('2024-02-07 00:00:00'), Timestamp('2024-01-15 00:00:00'), Timestamp('2024-02-10 00:00:00'), Timestamp('2024-02-12 00:00:00'), Timestamp('2024-02-14 00:00:00'), Times

In [ ]:
#Handle remnaining Nat values caused by MM_YY-DD format

nat_mask=df['order_date'].isnull()
df.loc[nat_mask,'order_date']=pd.to_datetime(
    raw_df.loc[nat_mask,'order_date'],dayfirst=True,errors='coerce'
)

print(nat_mask)



0     False
1     False
2     False
3     False
4     False
5      True
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13     True
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
Name: order_date, dtype: bool


In [ ]:
nat_count=df['order_date'].isnull().sum()
print(f"\nUnparsed dates remaining(NaT):{nat_count}")



df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
df['day_name']=df['order_date'].dt.strftime('%A')
df['day']=df['order_date'].dt.day
df['day_of_week']=df['order_date'].dt.day_name()


print("\n After Parsing Sample data\n\n\n\n")
print(df[['order_date','year','month','month_name','day_name','day','day_of_week']].head(8).to_string(index=False))


Unparsed dates remaining(NaT):0

 After Parsing Sample data




order_date  year  month month_name  day_name  day day_of_week
2024-01-05  2024      1    January    Friday    5      Friday
2024-01-07  2024      1    January    Sunday    7      Sunday
2024-01-08  2024      1    January    Monday    8      Monday
2024-01-10  2024      1    January Wednesday   10   Wednesday
2024-01-05  2024      1    January    Friday    5      Friday
2024-01-07  2024      1    January    Sunday    7      Sunday
2024-01-12  2024      1    January    Friday   12      Friday
2024-01-13  2024      1    January  Saturday   13    Saturday


In [ ]:
#Fix Standardize Customer names
print("Names before standardization")
print(df['customer_name'].unique()[:8].tolist())
#Notice :'AMIT VARMA'  ,'kiran mehta' ,'Ramesh Kumar' -all inconsistent



df['customer_name']=df['customer_name'].str.strip().str.title()
print("Names after standardization")
print(df['customer_name'].unique()[:8].tolist())

Names before standardization
['Ramesh Kumar', 'Priya Nair', 'AMIT VERMA', 'Sunita Patel', 'kiran mehta', 'Deepak Singh', 'UnknownCustomer', 'Ananya Das']
Names after standardization
['Ramesh Kumar', 'Priya Nair', 'Amit Verma', 'Sunita Patel', 'Kiran Mehta', 'Deepak Singh', 'Unknowncustomer', 'Ananya Das']


In [ ]:
#Category Fixing

print("Before\n\n\n")
print(df['category'])

#the data shows keyboards item lavbelled as 'Electronics ' but keyboard are

#step1:Build a boolean mask-true for rows where both conditon true
wrong_mask=(df['product']=='keyboard') &(df['category']=='Electronics')
print(f"Rows to fix:{wrong_mask.sum()}")
print("Before fix")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))

#Step2:use .loc to Only the rows where the maks is True
#df.loc[row_condition],'column_name']=new_value

df.loc[wrong_mask,'category']='Accessories'

print("\nAfter fix")
print(df[wrong_mask][['order_id','product','category']].to_string(index=False))
print("\nAll unique categories now:",sorted(df['category'].unique().tolist()))

Before



0       Electronics
1       Electronics
2       Accessories
3       Electronics
4       Electronics
5       Accessories
6       Electronics
7       Accessories
8       Electronics
9       Accessories
10      Electronics
11      Accessories
12      Electronics
13      Electronics
14      Accessories
15      Accessories
16      Accessories
17      Electronics
18      Electronics
19      Accessories
20      Electronics
21      Accessories
22      Electronics
23      Electronics
24      Accessories
25      Electronics
26    Uncategorized
27      Electronics
28      Accessories
29      Accessories
Name: category, dtype: object
Rows to fix:0
Before fix
Empty DataFrame
Columns: [order_id, product, category]
Index: []

After fix
Empty DataFrame
Columns: [order_id, product, category]
Index: []

All unique categories now: ['Accessories', 'Electronics', 'Uncategorized']


In [ ]:
print(df['category'])

0       Electronics
1       Electronics
2       Accessories
3       Electronics
4       Electronics
5       Accessories
6       Electronics
7       Accessories
8       Electronics
9       Accessories
10      Electronics
11      Accessories
12      Electronics
13      Electronics
14      Accessories
15      Accessories
16      Accessories
17      Electronics
18      Electronics
19      Accessories
20      Electronics
21      Accessories
22      Electronics
23      Electronics
24      Accessories
25      Electronics
26    Uncategorized
27      Electronics
28      Accessories
29      Accessories
Name: category, dtype: object


In [ ]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')

df['revenue']=df['quantity']*df['unit_price']
print(df[['quantity','unit_price','revenue']].head(8).to_string(index=False))



 quantity  unit_price  revenue
        2       45000    90000
        1       15000    15000
        3        1200     3600
        2       22000    44000
        2       45000    90000
       10         800     8000
        2        3500     7000
        1        2500     2500


In [ ]:
# ============================================================
# CELL 11 — Post-Cleaning Validation Report
# ============================================================


# Calculate the data quality score
# We check 5 things: no missing values, no duplicates, no date nulls, no revenue nulls
# Each passing check contributes 20 points (5 checks × 20 = 100)
missing_count   = df.isnull().sum().sum()
duplicate_count = df.duplicated().sum()
date_nulls      = df['order_date'].isnull().sum()
revenue_nulls   = df['revenue'].isnull().sum()


checks_passed   = sum([
    missing_count   == 0,   # 20 points
    duplicate_count == 0,   # 20 points
    date_nulls      == 0,   # 20 points
    revenue_nulls   == 0,   # 20 points
    len(df)         > 0     # 20 points (dataset is not empty)
])
quality_score = checks_passed * 20


# ── Print the report ─────────────────────────────────────────
print("=" * 55)
print("  POST-CLEANING VALIDATION REPORT")
print("=" * 55)
print(f"  Original rows   : {len(raw_df)}")
print(f"  Cleaned rows    : {len(df)}")
print(f"  Rows removed    : {len(raw_df) - len(df)} (duplicates)")
print(f"  Missing values  : {missing_count}")
print(f"  Duplicates      : {duplicate_count}")
print(f"  Date nulls      : {date_nulls}")
print(f"  Revenue nulls   : {revenue_nulls}")
print(f"  Columns total   : {len(df.columns)}")
print("=" * 55)
print(f"  DATA QUALITY SCORE : {quality_score}/100")
print(f"  DATA IS CLEAN      : {quality_score == 100}")
print("=" * 55)


# ── Actionable Debugging Suggestions ─────────────────────────
if missing_count > 0:
    print("\n  ACTION REQUIRED: Missing values detected.")
    print("  → Use df['column'].fillna(value, inplace=True)")
    print("  → For numbers: fillna(df['column'].median())")
    print("  → For text   : fillna('Unknown')")


if duplicate_count > 0:
    print("\n  ACTION REQUIRED: Duplicate rows detected.")
    print("  → Use df.drop_duplicates(inplace=True)")


if date_nulls > 0:
    print("\n  ACTION REQUIRED: Unparseable dates found.")
    print("  → Check for unusual date formats in the raw data")
    print("  → Use pd.to_datetime(col, dayfirst=True, errors='coerce')")


if quality_score == 100:
    print("\n  All checks passed. Data is ready for analysis.")

  POST-CLEANING VALIDATION REPORT
  Original rows   : 30
  Cleaned rows    : 30
  Rows removed    : 0 (duplicates)
  Missing values  : 0
  Duplicates      : 0
  Date nulls      : 0
  Revenue nulls   : 0
  Columns total   : 16
  DATA QUALITY SCORE : 100/100
  DATA IS CLEAN      : True

  All checks passed. Data is ready for analysis.


In [ ]:
output_filename = 'clean_data.csv'
df.to_csv(output_filename,index=False)
print(f"Cleaned data saved to {output_filename}")
print(f"final dataset's shape {df.shape}")

Cleaned data saved to clean_data.csv
final dataset's shape (30, 16)


In [ ]:
SERP_API_KEY='464ddd6c34f6425c8ad48ee1cbbebed05ee82f453a5f7075de7e376e2ece1239'
SERP_URL='https://serpapi.com/search.json'

SEARCH_QUERY='Data Enginner India'

print(f"SerpAPI Key  : {'Set (live data)' if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE' else 'Not set (fallback data will be used)'}")
print(f"Search query: {SEARCH_QUERY}")

SerpAPI Key  : Set (live data)
Search query: Data Enginner India


In [ ]:
def fetch_jobs(query, api_key, num_pages=2):
    """
    Fetches job listings from Google Jobs via SerpAPI.


    Parameters:
        query    (str) : The job search query (e.g. 'Data Engineer India')
        api_key  (str) : Your SerpAPI key
        num_pages(int) : Number of result pages to fetch (default: 2)


    Returns:
        list : A list of job dictionaries
    """
    all_jobs = []


    for page in range(num_pages):
        # API pagination: 'start' tells the API which result to start from
        # Page 0: results 0-9, Page 1: results 10-19, etc.
        params = {
            'engine'    : 'google_jobs',  # Use the Google Jobs search engine
            'q'         : query,
            'api_key'   : api_key,
            'hl'        : 'en',           # Language: English
            'start'     : page * 10       # Pagination offset
        }


        try:
            response = requests.get(SERP_URL, params=params, timeout=15)


            if response.status_code == 200:
                data = response.json()


                # 'jobs_results' is the key in the JSON that holds the job listings
                jobs = data.get('jobs_results', [])


                for job in jobs:
                    # Extract and normalize each job's fields
                    # .get('key', 'default') returns the value if the key exists,
                    # or 'default' if it does not — prevents KeyError crashes
                    all_jobs.append({
                        'title'      : job.get('title', 'Unknown Title'),
                        'company'    : job.get('company_name', 'Unknown Company'),
                        'location'   : job.get('location', 'Unknown Location'),
                        'posted'     : job.get('detected_extensions', {}).get('posted_at', 'Unknown'),
                        'salary'     : job.get('detected_extensions', {}).get('salary', 'Not Disclosed'),
                        'job_type'   : job.get('detected_extensions', {}).get('schedule_type', 'Not Specified'),
                        'description': job.get('description', '')[:300]  # First 300 characters only
                    })


                print(f"  Page {page + 1}: fetched {len(jobs)} jobs")
            else:
                print(f"  Page {page + 1}: API error {response.status_code}")


        except Exception as e:
            print(f"  Page {page + 1}: error — {e}")


    return all_jobs


# ── Actually call the function ────────────────────────────────
job_records = []


if SERP_API_KEY != 'YOUR_SERPAPI_KEY_HERE':
    print(f"Fetching job listings for: '{SEARCH_QUERY}'")
    job_records = fetch_jobs(SEARCH_QUERY, SERP_API_KEY)
    print(f"Total jobs fetched: {len(job_records)}")
else:
    print("No SerpAPI key provided — fallback job data will be loaded next.")




Fetching job listings for: 'Data Enginner India'
  Page 1: fetched 10 jobs
  Page 2: fetched 10 jobs
Total jobs fetched: 20


In [ ]:
df3 = pd.DataFrame(job_records)
df3.to_excel('job_records.xlsx',index=False)